# 🎵 Music Genre Classification: Inception-ResNet + Bi-GRU (CRNN)
## Target: ~94%+ on GTZAN | Audio → Mel Spectrogram → CNN → Bi-GRU → Softmax

**Architecture:**
- **Input**: Raw `.wav` / `.au` audio files from GTZAN
- **Stage 1**: Mel Spectrogram extraction (librosa)
- **Stage 2**: Inception-ResNet CNN blocks (spatial feature extraction)
- **Stage 3**: Bidirectional GRU (temporal pattern modeling)
- **Stage 4**: Dense + Softmax (10 genres)

**Kaggle Setup:**
1. Add the GTZAN dataset: *Settings → Add Data → Search "GTZAN Dataset Music Genre Classification"*
2. Enable GPU: *Settings → Accelerator → GPU T4 x2*
3. Run all cells!

## Step 0 — Runtime Check & Install Dependencies

In [ ]:
# Check GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '⚠️  No GPU detected — enable GPU in Settings!')

# Install / upgrade required libraries
!pip install -q librosa audiomentations torchinfo
print('\n✅ All packages ready')

## Step 1 — Locate GTZAN Dataset (Kaggle Path)

On Kaggle, datasets are mounted at `/kaggle/input/`. After adding the GTZAN dataset,
it will appear at `/kaggle/input/gtzan-dataset-music-genre-classification/`.

In [ ]:
import os, glob

# Auto-detect the genres folder inside Kaggle input
DATA_ROOT = None
candidates = [
    '/kaggle/input/gtzan-dataset-music-genre-classification/Data/genres_original',
    '/kaggle/input/gtzan-dataset-music-genre-classification/genres_original',
    '/kaggle/input/gtzan-dataset-music-genre-classification/genres',
    '/kaggle/input/gtzan-dataset-music-genre-classification',
]

# Also search dynamically in case the dataset slug differs
for root, dirs, files in os.walk('/kaggle/input'):
    audio_files = glob.glob(os.path.join(root, '**/*.wav'), recursive=True) + \
                  glob.glob(os.path.join(root, '**/*.au'),  recursive=True)
    if len(audio_files) > 500:
        candidates.insert(0, root)
        break

for candidate in candidates:
    audio_files = glob.glob(candidate + '/**/*.wav', recursive=True) + \
                  glob.glob(candidate + '/**/*.au',  recursive=True)
    if len(audio_files) > 500:
        DATA_ROOT = candidate
        break

assert DATA_ROOT is not None, '❌ Could not find GTZAN audio files. Did you add the dataset in Settings → Add Data?'
print(f'✅ DATA_ROOT = {DATA_ROOT}')

GENRES = sorted([d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d))])
print(f'Genres ({len(GENRES)}): {GENRES}')

total = sum(len(glob.glob(os.path.join(DATA_ROOT, g, '*'))) for g in GENRES)
print(f'Total audio files: {total}')

## Step 2 — Configuration

In [ ]:
import torch

class Config:
    # ── Audio ────────────────────────────────────────────────────────────────
    SAMPLE_RATE    = 22050
    DURATION       = 30          # seconds per track
    N_MELS         = 128         # mel bins
    HOP_LENGTH     = 512
    N_FFT          = 2048
    FMIN           = 20
    FMAX           = 8000

    # ── Segmentation (TTA via voting) ────────────────────────────────────────
    SEGMENT_SEC    = 5           # each segment duration
    SEGMENT_HOP    = 2.5         # hop between segments (50% overlap)

    # ── Training ─────────────────────────────────────────────────────────────
    BATCH_SIZE     = 32
    EPOCHS         = 80
    LR             = 3e-4
    WEIGHT_DECAY   = 1e-4
    LABEL_SMOOTH   = 0.1
    VAL_SPLIT      = 0.15        # 15% validation
    TEST_SPLIT     = 0.10        # 10% test  (NO leakage: split by track, not segment)
    SEED           = 42

    # ── Model ────────────────────────────────────────────────────────────────
    N_CLASSES      = 10
    GRU_HIDDEN     = 256
    GRU_LAYERS     = 2
    DROPOUT        = 0.4

    # ── Kaggle paths ─────────────────────────────────────────────────────────
    DEVICE         = 'cuda' if torch.cuda.is_available() else 'cpu'
    SAVE_PATH      = '/kaggle/working/best_model.pth'   # Kaggle output dir
    OUTPUT_DIR     = '/kaggle/working'

cfg = Config()
print(f'Device: {cfg.DEVICE}')
if cfg.DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Step 3 — Dataset: Leak-Free Split + Mel Spectrogram Extraction

In [ ]:
import numpy as np
import librosa
import random
from pathlib import Path
from collections import defaultdict

random.seed(cfg.SEED)
np.random.seed(cfg.SEED)
torch.manual_seed(cfg.SEED)

GENRE2IDX = {g: i for i, g in enumerate(GENRES)}
IDX2GENRE = {i: g for g, i in GENRE2IDX.items()}

# ── Gather all files grouped by genre ────────────────────────────────────────
genre_files = defaultdict(list)
for genre in GENRES:
    pattern_wav = os.path.join(DATA_ROOT, genre, '*.wav')
    pattern_au  = os.path.join(DATA_ROOT, genre, '*.au')
    files_found = sorted(glob.glob(pattern_wav) + glob.glob(pattern_au))
    # Remove the notorious corrupt file in GTZAN
    files_found = [f for f in files_found if 'jazz.00054' not in f]
    genre_files[genre] = files_found
    print(f'  {genre:12s}: {len(files_found)} files')

# ── Split BY TRACK (not segment) to prevent leakage ──────────────────────────
train_files, val_files, test_files = [], [], []
for genre, flist in genre_files.items():
    random.shuffle(flist)
    n = len(flist)
    n_test = max(1, int(n * cfg.TEST_SPLIT))
    n_val  = max(1, int(n * cfg.VAL_SPLIT))
    test_files  += [(f, GENRE2IDX[genre]) for f in flist[:n_test]]
    val_files   += [(f, GENRE2IDX[genre]) for f in flist[n_test:n_test+n_val]]
    train_files += [(f, GENRE2IDX[genre]) for f in flist[n_test+n_val:]]

print(f'\nTrack-level split (NO leakage):')
print(f'  Train: {len(train_files)} | Val: {len(val_files)} | Test: {len(test_files)}')

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import audiomentations as A

def load_audio(path, sr=cfg.SAMPLE_RATE, duration=cfg.DURATION):
    """Load audio, pad/trim to fixed duration."""
    target_len = sr * duration
    y, _ = librosa.load(path, sr=sr, mono=True, duration=duration)
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    else:
        y = y[:target_len]
    return y

def audio_to_mel(y, sr=cfg.SAMPLE_RATE):
    """Convert waveform → log-Mel spectrogram (128 × T)."""
    mel = librosa.feature.melspectrogram(
        y=y, sr=sr, n_mels=cfg.N_MELS,
        n_fft=cfg.N_FFT, hop_length=cfg.HOP_LENGTH,
        fmin=cfg.FMIN, fmax=cfg.FMAX
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    return mel_db.astype(np.float32)  # shape: (128, T)

def spec_augment(mel, freq_mask=20, time_mask=40, n_freq_masks=2, n_time_masks=2):
    """SpecAugment: frequency & time masking."""
    mel = mel.copy()
    T, F = mel.shape[1], mel.shape[0]
    for _ in range(n_freq_masks):
        f0 = random.randint(0, max(0, F - freq_mask))
        mel[f0:f0+random.randint(1, freq_mask), :] = mel.min()
    for _ in range(n_time_masks):
        t0 = random.randint(0, max(0, T - time_mask))
        mel[:, t0:t0+random.randint(1, time_mask)] = mel.min()
    return mel

# Audio augmentation pipeline (waveform-level)
train_augment = A.Compose([
    A.AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.015, p=0.4),
    A.TimeStretch(min_rate=0.85, max_rate=1.15, p=0.4),
    A.PitchShift(min_semitones=-3, max_semitones=3, p=0.4),
    A.Gain(min_gain_in_db=-6, max_gain_in_db=6, p=0.3),
], sample_rate=cfg.SAMPLE_RATE)


class GTZANDataset(Dataset):
    def __init__(self, file_label_pairs, augment=False):
        self.pairs   = file_label_pairs
        self.augment = augment

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        path, label = self.pairs[idx]
        y = load_audio(path)

        if self.augment:
            y = train_augment(samples=y, sample_rate=cfg.SAMPLE_RATE)

        mel = audio_to_mel(y)  # (128, T)

        if self.augment:
            mel = spec_augment(mel)

        # Normalize per-spectrogram
        mel = (mel - mel.mean()) / (mel.std() + 1e-8)

        # Add channel dim → (1, 128, T)
        mel = torch.tensor(mel, dtype=torch.float32).unsqueeze(0)
        return mel, label


train_ds = GTZANDataset(train_files, augment=True)
val_ds   = GTZANDataset(val_files,   augment=False)
test_ds  = GTZANDataset(test_files,  augment=False)

# Kaggle has 2 CPUs — use num_workers=2
train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Peek at shape
x, y = next(iter(train_loader))
print(f'Batch shape: {x.shape}  | Labels: {y.shape}')
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

## Step 4 — Model: Inception-ResNet CNN + Bi-GRU

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torchinfo import summary

class InceptionBlock(nn.Module):
    def __init__(self, in_ch, c1x1, c3x3_red, c3x3, c5x5_red, c5x5, pool_proj):
        super().__init__()
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_ch, c1x1, kernel_size=1, bias=False),
            nn.BatchNorm2d(c1x1), nn.GELU()
        )
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_ch, c3x3_red, kernel_size=1, bias=False),
            nn.BatchNorm2d(c3x3_red), nn.GELU(),
            nn.Conv2d(c3x3_red, c3x3, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(c3x3), nn.GELU()
        )
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_ch, c5x5_red, kernel_size=1, bias=False),
            nn.BatchNorm2d(c5x5_red), nn.GELU(),
            nn.Conv2d(c5x5_red, c5x5, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(c5x5), nn.GELU(),
            nn.Conv2d(c5x5, c5x5, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(c5x5), nn.GELU()
        )
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            nn.Conv2d(in_ch, pool_proj, kernel_size=1, bias=False),
            nn.BatchNorm2d(pool_proj), nn.GELU()
        )
        self.out_channels = c1x1 + c3x3 + c5x5 + pool_proj

    def forward(self, x):
        return torch.cat([self.branch1(x), self.branch2(x),
                          self.branch3(x), self.branch4(x)], dim=1)


class ResidualWrapper(nn.Module):
    def __init__(self, module, in_ch, out_ch):
        super().__init__()
        self.module = module
        self.shortcut = (
            nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False),
                nn.BatchNorm2d(out_ch)
            ) if in_ch != out_ch else nn.Identity()
        )

    def forward(self, x):
        return F.gelu(self.module(x) + self.shortcut(x))


class InceptionResNetCRNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.GELU(),
            nn.MaxPool2d(2, 2)
        )

        inc1a = InceptionBlock(64,  c1x1=32, c3x3_red=16, c3x3=48, c5x5_red=8,  c5x5=16, pool_proj=16)
        inc1b = InceptionBlock(112, c1x1=64, c3x3_red=32, c3x3=64, c5x5_red=16, c5x5=24, pool_proj=16)
        self.stage1 = nn.Sequential(
            ResidualWrapper(inc1a, 64,  inc1a.out_channels),
            ResidualWrapper(inc1b, inc1a.out_channels, inc1b.out_channels),
            nn.MaxPool2d(2, 2)
        )
        s1_out = inc1b.out_channels

        inc2a = InceptionBlock(s1_out, c1x1=64, c3x3_red=32, c3x3=96,  c5x5_red=16, c5x5=32, pool_proj=32)
        inc2b = InceptionBlock(inc2a.out_channels, c1x1=96, c3x3_red=48, c3x3=128, c5x5_red=24, c5x5=48, pool_proj=32)
        self.stage2 = nn.Sequential(
            ResidualWrapper(inc2a, s1_out, inc2a.out_channels),
            ResidualWrapper(inc2b, inc2a.out_channels, inc2b.out_channels),
            nn.MaxPool2d(2, 2)
        )
        s2_out = inc2b.out_channels

        inc3a = InceptionBlock(s2_out, c1x1=96, c3x3_red=48, c3x3=128, c5x5_red=24, c5x5=48, pool_proj=32)
        inc3b = InceptionBlock(inc3a.out_channels, c1x1=96, c3x3_red=48, c3x3=128, c5x5_red=24, c5x5=48, pool_proj=32)
        self.stage3 = nn.Sequential(
            ResidualWrapper(inc3a, s2_out, inc3a.out_channels),
            ResidualWrapper(inc3b, inc3a.out_channels, inc3b.out_channels),
        )
        s3_out = inc3b.out_channels

        self.freq_pool = nn.AdaptiveAvgPool2d((1, None))

        self.gru = nn.GRU(
            input_size  = s3_out,
            hidden_size = cfg.GRU_HIDDEN,
            num_layers  = cfg.GRU_LAYERS,
            batch_first = True,
            bidirectional=True,
            dropout     = cfg.DROPOUT if cfg.GRU_LAYERS > 1 else 0
        )

        gru_out = cfg.GRU_HIDDEN * 2
        self.attn = nn.Linear(gru_out, 1)

        self.classifier = nn.Sequential(
            nn.Dropout(cfg.DROPOUT),
            nn.Linear(gru_out, 256),
            nn.GELU(),
            nn.Dropout(cfg.DROPOUT / 2),
            nn.Linear(256, n_classes)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.freq_pool(x)
        x = x.squeeze(2).permute(0, 2, 1)
        x, _ = self.gru(x)
        attn_w = torch.softmax(self.attn(x), dim=1)
        x = (x * attn_w).sum(dim=1)
        return self.classifier(x)


model = InceptionResNetCRNN(n_classes=cfg.N_CLASSES).to(cfg.DEVICE)
summary(model, input_size=(2, 1, 128, 1293), device=cfg.DEVICE, depth=2)

## Step 5 — Training Setup: Label Smoothing + Cosine LR

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

criterion = nn.CrossEntropyLoss(label_smoothing=cfg.LABEL_SMOOTH)
optimizer = AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2, eta_min=1e-6)

print('Optimizer: AdamW')
print('Scheduler: CosineAnnealingWarmRestarts (T_0=20, T_mult=2)')
print('Loss: CrossEntropy + Label Smoothing 0.1')

## Step 6 — Training Loop

In [ ]:
from tqdm.notebook import tqdm

def run_epoch(loader, model, optimizer=None, train=True):
    model.train(train)
    total_loss, correct, total = 0, 0, 0
    with torch.set_grad_enabled(train):
        for X, y in tqdm(loader, leave=False,
                         desc='Train' if train else 'Val'):
            X, y = X.to(cfg.DEVICE), y.to(cfg.DEVICE)
            logits = model(X)
            loss   = criterion(logits, y)
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()
            total_loss += loss.item() * len(y)
            correct    += (logits.argmax(1) == y).sum().item()
            total      += len(y)
    return total_loss / total, correct / total


history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0

print(f'\n{"Epoch":>6} | {"Train Loss":>10} | {"Train Acc":>9} | {"Val Loss":>8} | {"Val Acc":>7} | LR')
print('─' * 65)

for epoch in range(1, cfg.EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, model, optimizer, train=True)
    vl_loss, vl_acc = run_epoch(val_loader,   model, train=False)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(vl_loss)
    history['val_acc'].append(vl_acc)

    lr_now = optimizer.param_groups[0]['lr']
    flag   = ' ★' if vl_acc > best_val_acc else ''
    print(f'{epoch:6d} | {tr_loss:10.4f} | {tr_acc:9.4f} | {vl_loss:8.4f} | {vl_acc:7.4f} | {lr_now:.2e}{flag}')

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), cfg.SAVE_PATH)

print(f'\n✅ Best val accuracy: {best_val_acc:.4f}')

## Step 7 — Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history['train_loss'], label='Train', color='#E8593C')
axes[0].plot(history['val_loss'],   label='Val',   color='#3B8BD4')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot([a*100 for a in history['train_acc']], label='Train', color='#E8593C')
axes[1].plot([a*100 for a in history['val_acc']],   label='Val',   color='#3B8BD4')
axes[1].axhline(y=best_val_acc*100, color='gray', linestyle='--', alpha=0.5,
                label=f'Best val: {best_val_acc*100:.1f}%')
axes[1].set_title('Accuracy (%)'); axes[1].set_xlabel('Epoch'); axes[1].legend()

plt.tight_layout()
# Saved to Kaggle output directory
plt.savefig(os.path.join(cfg.OUTPUT_DIR, 'training_curves.png'), dpi=120)
plt.show()
print(f'✅ Saved to {cfg.OUTPUT_DIR}/training_curves.png')

## Step 8 — Test Evaluation with Segment Voting (TTA)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Load best checkpoint
model.load_state_dict(torch.load(cfg.SAVE_PATH, map_location=cfg.DEVICE))
model.eval()

# ── Standard evaluation ──────────────────────────────────────────────────────
all_preds, all_labels = [], []
with torch.no_grad():
    for X, y in tqdm(test_loader, desc='Test eval'):
        logits = model(X.to(cfg.DEVICE))
        all_preds.extend(logits.argmax(1).cpu().tolist())
        all_labels.extend(y.tolist())

acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
print(f'\n🎯 Test Accuracy (single-pass): {acc*100:.2f}%\n')
print(classification_report(all_labels, all_preds, target_names=GENRES))

In [ ]:
# ── Test-Time Augmentation via segment voting ────────────────────────────────
def predict_with_voting(audio_path, model, n_segments=10):
    y_full = load_audio(audio_path)
    seg_len = int(cfg.SAMPLE_RATE * cfg.SEGMENT_SEC)
    hop_len = int(cfg.SAMPLE_RATE * cfg.SEGMENT_HOP)
    probs = []
    for start in range(0, len(y_full) - seg_len + 1, hop_len):
        seg = y_full[start:start + seg_len]
        mel = audio_to_mel(seg)
        mel = (mel - mel.mean()) / (mel.std() + 1e-8)
        mel_t = torch.tensor(mel).unsqueeze(0).unsqueeze(0).to(cfg.DEVICE)
        with torch.no_grad():
            logit = model(mel_t)
            prob  = torch.softmax(logit, dim=1).squeeze(0).cpu().numpy()
        probs.append(prob)
    return np.stack(probs).sum(axis=0).argmax()

print('Running TTA segment voting on test set ...')
tta_preds, tta_labels = [], []
for path, label in tqdm(test_files):
    pred = predict_with_voting(path, model)
    tta_preds.append(pred)
    tta_labels.append(label)

tta_acc = sum(p == l for p, l in zip(tta_preds, tta_labels)) / len(tta_labels)
print(f'\n🎯 Test Accuracy (TTA segment voting): {tta_acc*100:.2f}%')

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(tta_labels, tta_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=GENRES, yticklabels=GENRES)
plt.title(f'Confusion Matrix — TTA Accuracy: {tta_acc*100:.1f}%')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig(os.path.join(cfg.OUTPUT_DIR, 'confusion_matrix.png'), dpi=120)
plt.show()
print(f'✅ Saved to {cfg.OUTPUT_DIR}/confusion_matrix.png')

## Step 9 — Save Model + Genre Metadata for Streamlit App

This saves `best_model.pth` and `genres.json` to `/kaggle/working/`.
Download both files from the **Output** tab to use with the Streamlit app.

In [ ]:
import json

# Save genre mapping for the Streamlit app
metadata = {
    'genres': GENRES,
    'genre2idx': GENRE2IDX,
    'idx2genre': {str(k): v for k, v in IDX2GENRE.items()},
    'n_classes': cfg.N_CLASSES,
    'sample_rate': cfg.SAMPLE_RATE,
    'n_mels': cfg.N_MELS,
    'hop_length': cfg.HOP_LENGTH,
    'n_fft': cfg.N_FFT,
    'fmin': cfg.FMIN,
    'fmax': cfg.FMAX,
    'segment_sec': cfg.SEGMENT_SEC,
    'segment_hop': cfg.SEGMENT_HOP,
    'gru_hidden': cfg.GRU_HIDDEN,
    'gru_layers': cfg.GRU_LAYERS,
    'dropout': cfg.DROPOUT,
    'best_val_acc': best_val_acc,
    'tta_test_acc': tta_acc,
}

with open(os.path.join(cfg.OUTPUT_DIR, 'genres.json'), 'w') as f:
    json.dump(metadata, f, indent=2)

print('✅ Files saved to /kaggle/working/')
print('   → best_model.pth')
print('   → genres.json')
print('   → training_curves.png')
print('   → confusion_matrix.png')
print('\nDownload these from the Output tab and place them in your Streamlit app folder.')

---
## Architecture Summary

```
Audio (.wav / .au)
    │
    ▼  librosa
Mel Spectrogram  (128 mel bins × T frames, log-scale)
    │
    ▼  Data Augmentation (noise, pitch, stretch, SpecAugment)
    │
    ▼  Conv Stem  1 → 64 channels, MaxPool
    │
    ▼  Stage 1  Inception-ResNet × 2  → 168ch, MaxPool
    │
    ▼  Stage 2  Inception-ResNet × 2  → 304ch, MaxPool
    │
    ▼  Stage 3  Inception-ResNet × 2  → 304ch
    │
    ▼  AdaptiveAvgPool (collapse freq → 1)  → sequence (B, T', 304)
    │
    ▼  Bidirectional GRU  ×2 layers, hidden=256  → (B, T', 512)
    │
    ▼  Temporal Attention Pooling  → (B, 512)
    │
    ▼  Dropout → FC(256) → GELU → Dropout → FC(10)
    │
    ▼  Softmax
10 Genres: blues classical country disco hiphop jazz metal pop reggae rock
```